<div align='center'>

# ⚡ KRONOS v2 — Orbital Strategist
### *Seven Weapons Nobody Has Combined Before*

</div>

---

| # | Weapon | Core Idea |
|---|--------|-----------|
| 1 | **Desynchronization Attack** | Multi-wave from different angles → same arrival turn → defender overwhelmed |
| 2 | **Economic Suffocation** | 3-ship squads arrive every 15 turns → enemy production counter never builds up |
| 3 | **Snipe Protocol** | Arrive 2 turns after enemy weakens a neutral → capture it at 0 defense |
| 4 | **Pre-emptive Strike** | Detect enemy massing ships → attack BEFORE they launch |
| 5 | **Multi-Enemy Awareness** | Sum nearby enemy power (weighted by proximity), not just strongest |
| 6 | **Orbital Phase Windows** | Attack inner planets when they're moving toward us → 1.65× efficiency |
| 7 | **Retrograde Warfare** | Counter-attack emptied source planet the moment enemy fleet departs |

---


## ⚙️ Cell 1 — Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'


## 🔌 Cell 2 — Environment


In [ ]:
from kaggle_environments import make
import math, collections

env = make('orbit_wars', debug=True)
print(f'✅ {env.name} v{env.version} | steps={env.configuration.episodeSteps}')
s = env.reset()[0]['observation']
print(f'Planet[0]: {s["planets"][0]}')
print(f'av={s["angular_velocity"]:.4f}')


## ⚡ Cell 3 — KRONOS v2 Agent

Seven weapons in one cell. This is the submission function.


In [ ]:
"""
KRONOS v2 — "God of Time & Strategy"
Seven weapons nobody has combined before:
  1. Desynchronization Attack  — multi-wave same-turn arrival from different angles
  2. Economic Suffocation      — harassment squads reset enemy production counters  
  3. Snipe Protocol            — arrive 1 turn after enemy weakens a neutral
  4. Pre-emptive Strike        — attack when enemy is massing, before they launch
  5. Multi-Enemy Awareness     — sum nearby enemy power, not just strongest
  6. Orbital Phase Windows     — time attacks when inner planets move toward us
  7. Retrograde Warfare        — counter-attack emptied source planets
"""
import math, collections

SX, SY, SR, INNER, MS = 50.0, 50.0, 5.0, 38.0, 500

# ── Wrappers ──────────────────────────────────────────────────────────────────
class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

# ── Physics ───────────────────────────────────────────────────────────────────
def spd(n):   return min(6.0, 1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 1: DESYNCHRONIZATION ATTACK
# Split N ships into 2-3 waves from different source planets
# All timed to arrive at target within 2 turns of each other
# Defender AI calculates garrison only for first wave → rest overwhelm
# ─────────────────────────────────────────────────────────────────────────────
def desync_attack(sources, target, av, used):
    """
    Try to send 2-3 waves to arrive simultaneously.
    Returns list of moves if feasible, else [].
    """
    if len(sources) < 2: return []
    
    # Calculate arrival time from each source with minimum ships
    arrivals = []
    for src in sources[:3]:
        spare = src.ships - used.get(src.id,0) - 3
        if spare < 4: continue
        _,dd,eta = icp(src.x,src.y,target,av,spare)
        arrivals.append((eta, src, spare, dd))
    
    if len(arrivals) < 2: return []
    
    # Find target arrival time = median
    arrivals.sort(key=lambda x: x[0])
    target_eta = arrivals[len(arrivals)//2][0]
    
    # For each source, compute ships needed to arrive at target_eta
    # If eta < target_eta: send fewer ships (slower fleet)
    # If eta > target_eta: can't slow down, skip
    moves = []
    total_sent = 0
    garrison_arr = target.ships + target.production * target_eta
    
    for eta, src, spare, dd in arrivals:
        if eta > target_eta + 3: continue  # too far, skip
        
        # Ships to send from this source
        frac = spare // len(arrivals)
        n = max(4, frac)
        if src.ships - used.get(src.id,0) - n < 3: continue
        
        a2,dd2,_ = icp(src.x,src.y,target,av,n)
        sa,ok = safe(src.x,src.y,a2,dd2)
        if not ok: continue
        
        moves.append((src.id, sa, n))
        total_sent += n
    
    # Only return if combined force is enough to capture
    if total_sent > garrison_arr * 1.05 and len(moves) >= 2:
        return moves
    return []

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 2: ECONOMIC SUFFOCATION
# Send tiny fleets (2-4 ships) to arrive at enemy high-production planets
# exactly when their production counter resets → forces them to keep garrison
# Result: enemy planet never accumulates ships to attack us
# ─────────────────────────────────────────────────────────────────────────────
def suffocation_targets(enemy_planets, mine, av, used, step):
    """
    Find enemy planets worth suffocating.
    Returns list of (source, target, n_ships, safe_angle)
    """
    if step < 60: return []  # don't suffocate in early game
    
    moves = []
    high_prod = sorted([p for p in enemy_planets if p.production >= 3],
                       key=lambda p: -p.production)[:2]
    
    for tgt in high_prod:
        # Find closest source with spare ships
        best_src = None; best_dd = 1e9
        for src in mine:
            spare = src.ships - used.get(src.id,0) - 5
            if spare < 3: continue
            dd = d2(src.x,src.y,tgt.x,tgt.y)
            if dd < best_dd:
                best_dd = dd; best_src = src
        
        if best_src is None: continue
        
        # Send just 3 ships — enough to force garrison, not enough to capture
        # Arrive every ~15 turns (production cycle)
        if step % 15 != 0 and step % 15 != 1: continue
        
        n = 3
        a,dd,_ = icp(best_src.x,best_src.y,tgt,av,n)
        sa,ok = safe(best_src.x,best_src.y,a,dd)
        if ok and best_src.ships - used.get(best_src.id,0) - n >= 5:
            moves.append((best_src.id, sa, n))
    
    return moves

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 3: SNIPE PROTOCOL
# Detect enemy fleets heading to neutral planets
# Send our fleet to arrive 1-2 turns AFTER enemy lands → they weakened it for us
# ─────────────────────────────────────────────────────────────────────────────
def find_snipe_opportunities(neutral_planets, enemy_fleets, mine, av, used):
    """
    For each neutral planet with an incoming enemy fleet:
    - Estimate enemy arrival time
    - Estimate post-battle state (enemy ships - garrison)
    - If we can arrive 1-3 turns after with enough to capture → SNIPE
    """
    snipes = []
    
    for neutral in neutral_planets:
        # Find enemy fleets heading here
        incoming = []
        for f in enemy_fleets:
            _,dd,eta = icp(f.x,f.y,neutral,av,f.ships)
            if dd < neutral.radius + 4 and eta < 50:
                incoming.append((eta, f.ships))
        
        if not incoming: continue
        
        incoming.sort(key=lambda x: x[0])
        enemy_eta, enemy_ships = incoming[0]
        
        # After battle: neutral has (neutral.ships + prod*eta) - enemy_ships survivors
        # If enemy_ships > neutral, enemy captures it with leftover ships
        after_neutral = neutral.ships + neutral.production * enemy_eta
        if enemy_ships <= after_neutral:
            continue  # enemy loses, neutral survives at full strength — skip
        
        # Enemy captures neutral with (enemy_ships - after_neutral) ships
        enemy_leftover = max(1, enemy_ships - after_neutral)
        
        # We arrive 2 turns later → need to beat enemy_leftover + 2 turns production
        snipe_eta = enemy_eta + 2
        ships_needed = int(enemy_leftover * 1.1) + neutral.production * 2 + 2
        
        # Find best source
        for src in sorted(mine, key=lambda p: d2(p.x,p.y,neutral.x,neutral.y)):
            spare = src.ships - used.get(src.id,0) - 4
            if spare < ships_needed: continue
            
            a,dd,our_eta = icp(src.x,src.y,neutral,av,ships_needed)
            # We need our_eta ≈ snipe_eta (arrive after enemy)
            if our_eta < enemy_eta + 0.5: continue  # we'd arrive too early
            if our_eta > enemy_eta + 8:   continue  # too late
            
            sa,ok = safe(src.x,src.y,a,dd)
            if ok:
                score = neutral.production * 5 + (enemy_ships - enemy_leftover)
                snipes.append((score, src.id, sa, ships_needed, neutral.id))
                break
    
    snipes.sort(key=lambda x: -x[0])
    return snipes

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 4: PRE-EMPTIVE STRIKE
# If enemy planet has > 80% of ships it had 5 turns ago → it's massing
# Attack NOW before they launch → destroy ships in harbor
# ─────────────────────────────────────────────────────────────────────────────
_ship_history = collections.defaultdict(list)  # planet_id -> [ships_count]

def update_history(planets):
    for p in planets:
        if p.owner >= 0:
            _ship_history[p.id].append(p.ships)
            if len(_ship_history[p.id]) > 8:
                _ship_history[p.id].pop(0)

def find_massing_planets(enemy_planets, mine, av, used):
    """
    Detect enemy planets rapidly accumulating ships.
    Returns (planet, urgency) pairs sorted by urgency.
    """
    massing = []
    for p in enemy_planets:
        hist = _ship_history.get(p.id, [])
        if len(hist) < 4: continue
        
        growth = p.ships - hist[-4]  # ships gained in last 4 turns
        expected_growth = p.production * 4
        
        # If growing faster than production alone → they're NOT attacking → massing
        if growth > expected_growth * 0.7 and p.ships > 30:
            urgency = p.ships * p.production
            massing.append((p, urgency))
    
    massing.sort(key=lambda x: -x[1])
    return massing

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 5: MULTI-ENEMY AWARENESS (improved dominance)
# Count nearby enemy power, not just global strongest
# ─────────────────────────────────────────────────────────────────────────────
def compute_status(planets, fleets, mine, player):
    """
    Returns ('winning'/'losing', aggression_level 0-1)
    aggression_level: 0 = turtle, 1 = all-in
    """
    if not mine: return 'losing', 1.0
    
    my_cx = sum(p.x for p in mine)/len(mine)
    my_cy = sum(p.y for p in mine)/len(mine)
    
    my_ships = sum(p.ships for p in mine)
    my_prod  = sum(p.production for p in mine)
    my_fleet = sum(f.ships for f in fleets if f.owner==player)
    my_power = my_ships + my_prod*25 + my_fleet
    
    # Sum nearby enemy power (within 45 units of our centroid)
    nearby_enemy_power = 0
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        proximity = max(0, 1 - d2(p.x,p.y,my_cx,my_cy)/60)
        nearby_enemy_power += (p.ships + p.production*25) * (0.5+proximity)
    
    # Add enemy fleets heading toward us
    for f in fleets:
        if f.owner==player or f.owner<0: continue
        # Check if heading roughly toward our centroid
        dx,dy = math.cos(f.angle),math.sin(f.angle)
        dot = (my_cx-f.x)*dx + (my_cy-f.y)*dy
        if dot > 0:
            nearby_enemy_power += f.ships * 0.8
    
    ratio = my_power / max(1, nearby_enemy_power)
    
    if ratio >= 1.15:
        return 'winning', 0.5   # winning: moderate aggression
    elif ratio >= 0.85:
        return 'even', 0.75     # even: push harder
    else:
        return 'losing', 1.0    # losing: all-in, break their economy

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 6: ORBITAL PHASE WINDOWS
# ─────────────────────────────────────────────────────────────────────────────
def phase_mult(planet, mine, av):
    if not inn(planet) or not mine: return 1.0
    r  = d2(planet.x,planet.y,SX,SY)
    a0 = math.atan2(planet.y-SY,planet.x-SX)
    a15= a0+av*15
    fx = SX+r*math.cos(a15); fy=SY+r*math.sin(a15)
    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)
    df = d2(fx,fy,cx,cy); dc=d2(planet.x,planet.y,cx,cy)
    if df < dc*0.85: return 1.65
    if df > dc*1.15: return 0.72
    return 1.0

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 7: RETROGRADE WARFARE
# ─────────────────────────────────────────────────────────────────────────────
def retro_targets(planets, fleets, player):
    retro = []
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        departed = sum(f.ships for f in fleets
                       if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<12 and f.ships>0)
        if departed < 8: continue
        ratio = departed/max(1,p.ships+departed)
        if ratio >= 0.28:
            retro.append((p, ratio*p.production*4+departed*0.3))
    retro.sort(key=lambda x:-x[1])
    return retro

# ─────────────────────────────────────────────────────────────────────────────
# GARRISON
# ─────────────────────────────────────────────────────────────────────────────
def garrison(planet, status, aggression, incoming=0):
    if incoming>0: return int(incoming*1.12)+5
    base = max(3, planet.production*2)
    if status=='winning':   return base
    if aggression > 0.85:   return max(3, base//2)  # losing: bare minimum
    return base

# ─────────────────────────────────────────────────────────────────────────────
# MAIN AGENT
# ─────────────────────────────────────────────────────────────────────────────
def orbital_strategist(obs):
    global _ship_history
    
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    = [p for p in planets if p.owner==pl]
    neutral = [p for p in planets if p.owner<0]
    enemy   = [p for p in planets if p.owner>=0 and p.owner!=pl]
    others  = enemy+neutral

    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()

    def avail(p): return p.ships-used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n

    # ── Update massing history ─────────────────────────────────────────────
    update_history(planets)

    # ── Compute status ─────────────────────────────────────────────────────
    status, aggr = compute_status(planets, fleets, mine, pl)

    # ── Incoming threats ───────────────────────────────────────────────────
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_ = icp(f.x,f.y,p,av,f.ships)
            if dd < p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships

    # ── DEFENSE ───────────────────────────────────────────────────────────
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=garrison(p,status,aggr,thr); deficit=need-avail(p)
        if deficit<=0: continue
        donors=sorted([s for s in mine if s.id!=p.id and
                       avail(s)-garrison(s,status,aggr)>4],
                      key=lambda s:d2(s.x,s.y,p.x,p.y))
        for src in donors[:3]:
            send=min(avail(src)-garrison(src,status,aggr),deficit)
            if send<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,send); sa,ok=safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,send]); rsv(src.id,send); deficit-=send
            if deficit<=0: break

    # ── En-route targets ───────────────────────────────────────────────────
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<70: enroute.add(t.id)

    enemy_fleets=[f for f in fleets if f.owner!=pl and f.owner>=0]

    # ── WEAPON 3: SNIPE PROTOCOL ──────────────────────────────────────────
    snipes=find_snipe_opportunities(
        [t for t in neutral if t.id not in done and t.id not in enroute],
        enemy_fleets, mine, av, used)
    for score,sid,sa,n,tid in snipes[:2]:
        src=next((p for p in mine if p.id==sid),None)
        if src is None or avail(src)<n+garrison(src,status,aggr): continue
        moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── WEAPON 4: PRE-EMPTIVE STRIKE ──────────────────────────────────────
    massing=find_massing_planets(enemy,mine,av,used)
    for tgt,urg in massing[:1]:
        if tgt.id in done or tgt.id in enroute: continue
        best_src=max(
            [p for p in mine if avail(p)-garrison(p,status,aggr)>8],
            key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if best_src is None: continue
        spare=avail(best_src)-garrison(best_src,status,aggr)
        _,_,eta=icp(best_src.x,best_src.y,tgt,av,spare)
        n=max(int((tgt.ships+tgt.production*eta)*1.08)+1,tgt.ships+3)
        if spare<n: continue
        a,dd,_=icp(best_src.x,best_src.y,tgt,av,n)
        sa,ok=safe(best_src.x,best_src.y,a,dd)
        if ok:
            moves.append([best_src.id,sa,n]); rsv(best_src.id,n); done.add(tgt.id)

    # ── WEAPON 7: RETROGRADE ──────────────────────────────────────────────
    for rp_t,rscore in retro_targets(planets,fleets,pl)[:2]:
        if rp_t.id in done or rp_t.id in enroute: continue
        best_src=None; best_n=0; best_sa=0.0; best_dd=1e9
        for src in mine:
            spare=avail(src)-garrison(src,status,aggr)
            if spare<4: continue
            _,_,eta=icp(src.x,src.y,rp_t,av,spare)
            n=max(int((rp_t.ships+rp_t.production*eta)*1.06)+1,rp_t.ships+2)
            if spare<n: continue
            a2,dd2,_=icp(src.x,src.y,rp_t,av,n); sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if best_src is None or dd2<best_dd:
                best_src,best_n,best_sa,best_dd=src,n,sa,dd2
        if best_src and avail(best_src)-garrison(best_src,status,aggr)>=best_n:
            moves.append([best_src.id,best_sa,best_n])
            rsv(best_src.id,best_n); done.add(rp_t.id)

    # ── WEAPON 1: DESYNC ATTACK on high-value targets ─────────────────────
    high_value=[t for t in enemy if t.id not in done and t.id not in enroute
                and t.production>=3 and t.ships>20]
    for tgt in sorted(high_value,key=lambda t:-t.production*t.ships)[:1]:
        srcs=[p for p in mine if avail(p)-garrison(p,status,aggr)>6]
        if len(srcs)>=2:
            dm=desync_attack(srcs,tgt,av,used)
            if dm:
                for sid,sa,n in dm:
                    moves.append([sid,sa,n]); rsv(sid,n)
                done.add(tgt.id)

    # ── WEAPON 2: SUFFOCATION ─────────────────────────────────────────────
    suf=suffocation_targets(enemy,mine,av,used,stp)
    for sid,sa,n in suf:
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── MAIN SCORING: all remaining targets ───────────────────────────────
    candidates=[]
    for src in mine:
        spare=avail(src)-garrison(src,status,aggr)
        if spare<4: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            _,_,eta=icp(src.x,src.y,tgt,av,max(spare,5))
            buf=1.08 if tgt.owner>=0 else 1.05
            n=max(int((tgt.ships+tgt.production*eta)*buf)+1,int(tgt.ships*buf)+2)
            if n>spare: continue
            a2,dd2,eta2=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            tw=max(0,rem-eta2); prod=tgt.production
            score=(prod**2)*10*tw+prod*tw
            score*=phase_mult(tgt,mine,av)
            if tgt.owner>=0:
                score*=1.5
                e_prod=sum(p.production for p in planets if p.owner==tgt.owner)
                if e_prod>sum(p.production for p in mine)*1.1: score*=1.3
            if tgt.ships<=tgt.production*2+3: score*=1.9
            score-=dd2*0.4+n*0.25
            if status=='losing': score=score*1.4 if prod>=3 else score*0.6
            candidates.append((score,src,tgt,n,sa,dd2))

    candidates.sort(key=lambda x:-x[0])
    max_atk=5 if (stp<90 or status=='losing') else 4
    atks=0
    for score,src,tgt,n,sa,dd in candidates:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        spare=avail(src)-garrison(src,status,aggr)
        if spare<n: continue
        moves.append([src.id,sa,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ─────────────────────────────────────────────
    for src in sorted(mine,key=lambda p:-avail(p)):
        spare=avail(src)-garrison(src,status,aggr)
        if spare<5: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            _,_,eta=icp(src.x,src.y,tgt,av,spare)
            buf=1.08 if tgt.owner>=0 else 1.05
            n=max(int((tgt.ships+tgt.production*eta)*buf)+1,int(tgt.ships*buf)+2)
            if n>spare: continue
            a2,dd2,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            sc=(tgt.production**2)/(dd2+1)*phase_mult(tgt,mine,av)+spare*0.05
            if sc>bsc: bsc=sc; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## 🔬 Cell 4 — v1 Baseline Agent


In [ ]:
def v1_agent(obs):
    import math
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P2, Fleet as _F2
    except:
        class _P2:
            __slots__=['id','owner','x','y','radius','ships','production']
            def __init__(self,*a):
                for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
        class _F2:
            __slots__=['id','owner','x','y','angle','ships']
            def __init__(self,*a):
                for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def fs(n): return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
    def dd(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
    def isin(p): return dd(p.x,p.y,50,50)<38
    def pp(p,av,t):
        if not isin(p): return p.x,p.y
        r=dd(p.x,p.y,50,50);a=math.atan2(p.y-50,p.x-50)+av*t
        return 50+r*math.cos(a),50+r*math.sin(a)
    def icp2(sx,sy,tp,av,n):
        tx,ty=tp.x,tp.y
        for _ in range(15):
            d=dd(sx,sy,tx,ty);t=d/fs(n) if fs(n)>0 else 1e9;nx,ny=pp(tp,av,t)
            if dd(tx,ty,nx,ny)<0.05: break
            tx,ty=nx,ny
        d=dd(sx,sy,tx,ty);return math.atan2(ty-sy,tx-sx),tx,ty,d,d/fs(n) if fs(n)>0 else 1e9
    def sh(ox,oy,a,md2):
        dx,dy=math.cos(a),math.sin(a);fx,fy=50-ox,50-oy;t=fx*dx+fy*dy
        return 0<t<md2 and abs(fx*dy-fy*dx)<6.5
    def sa2(ox,oy,a,d2):
        if not sh(ox,oy,a,d2): return a,True
        for i in range(1,13):
            dl=math.radians(25)*i/12
            for s in(1,-1):
                if not sh(ox,oy,a+s*dl,d2): return a+s*dl,True
        return a,False
    if isinstance(obs,dict):
        player=obs.get('player',0);rp=obs.get('planets',[]);rf=obs.get('fleets',[])
        av=obs.get('angular_velocity',0.0366);step=obs.get('step',250)
    else:
        player=obs.player;rp=obs.planets;rf=obs.fleets;av=obs.angular_velocity;step=getattr(obs,'step',250)
    planets=[_P2(*p) for p in rp];fleets=[_F2(*f) for f in rf]
    mine=[p for p in planets if p.owner==player];tgts=[p for p in planets if p.owner!=player]
    if not mine or not tgts: return []
    rem=500-step;moves=[];committed=set();used={}
    def av2(p): return p.ships-used.get(p.id,0)
    for t in sorted([t for t in tgts if t.id not in committed and t.ships<=t.production*4+3],key=lambda t:t.ships):
        bst=None;bs=-1e9
        for src in mine:
            if av2(src)<15: continue
            _,_,_,ddv,ta=icp2(src.x,src.y,t,av,t.ships+5);sc=t.production/(ddv+1)
            if sc>bs: bs=sc;bst=src;bta=ta
        if bst is None: continue
        n=max(int((t.ships+t.production*bta)*1.3)+1,int(t.ships*1.3)+5)
        if av2(bst)<n: continue
        ang,_,_,ddv,_=icp2(bst.x,bst.y,t,av,n);sva,ok=sa2(bst.x,bst.y,ang,ddv)
        if ok: moves.append([bst.id,sva,n]);used[bst.id]=used.get(bst.id,0)+n;committed.add(t.id)
    cands=[]
    for src in mine:
        a2v=av2(src)
        if a2v<10: continue
        for t in tgts:
            if t.id in committed: continue
            _,_,_,ddv,ta=icp2(src.x,src.y,t,av,min(a2v,50))
            n=max(int((t.ships+t.production*ta)*1.3)+1,int(t.ships*1.3)+5)
            if n>a2v: continue
            g=t.ships+t.production*ta
            if n<=g: continue
            r=(t.production*max(0,rem-ta)-n)/(ta+1)+t.production*2
            if t.ships<=t.production*4+3: r*=1.5
            cands.append((r,src,t,n,ddv))
    cands.sort(key=lambda x:-x[0])
    for r,src,t,n,ddv in cands:
        if t.id in committed or av2(src)<n: continue
        ang,_,_,dd2,_=icp2(src.x,src.y,t,av,n);sva,ok=sa2(src.x,src.y,ang,dd2)
        if not ok: continue
        moves.append([src.id,sva,n]);used[src.id]=used.get(src.id,0)+n;committed.add(t.id)
    return moves

print('✅ v1 baseline ready')


## 🧪 Cell 5 — Test: KRONOS v2 vs v1 (1v1)


In [ ]:
import collections as _c
_ship_history = _c.defaultdict(list)  # reset for clean test

e1 = make('orbit_wars', debug=False)
e1.run([orbital_strategist, v1_agent])
r1 = [s.reward for s in e1.steps[-1]]
print(f'{'🏆' if r1[0]==1 else '  '} KRONOS v2 : {r1[0]:+d}')
print(f'{'🏆' if r1[1]==1 else '  '} v1        : {r1[1]:+d}')
e1.render(mode='ipython', width=800, height=600)


## 🎮 Cell 6 — 4-Player Showdown


In [ ]:
_ship_history = _c.defaultdict(list)
e4 = make('orbit_wars', debug=False)
e4.run([orbital_strategist, v1_agent, 'random', v1_agent])
r4 = [s.reward for s in e4.steps[-1]]
labels = ['⚡ KRONOS v2', 'v1-A', '🎲 Random', 'v1-B']
for lb,rw in zip(labels,r4):
    print(f'  {'🏆' if rw==1 else '  '} {lb:14s}  {rw:+d}')
e4.render(mode='ipython', width=800, height=600)


## 📊 Cell 7 — Tournament 20 Games (Randomized Positions)


In [ ]:
import random as _rnd

N = 20
wins = {'KRONOS':0,'v1':0,'random':0}

for g in range(N):
    _ship_history = _c.defaultdict(list)  # fresh history each game
    agents = [orbital_strategist, v1_agent, 'random', v1_agent]
    _rnd.shuffle(agents)
    kp = agents.index(orbital_strategist)
    et = make('orbit_wars', debug=False)
    et.run(agents)
    rws = [s.reward for s in et.steps[-1]]
    w = rws.index(max(rws))
    if w == kp:                       wins['KRONOS']+=1; wl='⚡ KRONOS v2'
    elif agents[w] == v1_agent:       wins['v1']+=1;    wl='   v1'
    else:                             wins['random']+=1; wl='🎲 Random'
    print(f'G{g+1:02d} [K@{kp}] {[f"{r:+d}" for r in rws]} → {wl}')

print('─'*50)
for nm,w in wins.items():
    print(f'  {nm:8s}: {w:2d}/{N}  {"█"*(w*2)}')
wr=wins['KRONOS']/N
elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}')
print(f'  Elo est. : ~{elo}')
print(f'  {"🏆 TOP 3 POTENTIAL!" if elo>=1400 else "✅ Competitive" if elo>=1000 else "⚠️ Needs work"}')


## 💾 Cell 8 — Write Submission (`main.py`)


In [ ]:
%%writefile main.py
"""
KRONOS v2 — "God of Time & Strategy"
Seven weapons nobody has combined before:
  1. Desynchronization Attack  — multi-wave same-turn arrival from different angles
  2. Economic Suffocation      — harassment squads reset enemy production counters  
  3. Snipe Protocol            — arrive 1 turn after enemy weakens a neutral
  4. Pre-emptive Strike        — attack when enemy is massing, before they launch
  5. Multi-Enemy Awareness     — sum nearby enemy power, not just strongest
  6. Orbital Phase Windows     — time attacks when inner planets move toward us
  7. Retrograde Warfare        — counter-attack emptied source planets
"""
import math, collections

SX, SY, SR, INNER, MS = 50.0, 50.0, 5.0, 38.0, 500

# ── Wrappers ──────────────────────────────────────────────────────────────────
class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

# ── Physics ───────────────────────────────────────────────────────────────────
def spd(n):   return min(6.0, 1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 1: DESYNCHRONIZATION ATTACK
# Split N ships into 2-3 waves from different source planets
# All timed to arrive at target within 2 turns of each other
# Defender AI calculates garrison only for first wave → rest overwhelm
# ─────────────────────────────────────────────────────────────────────────────
def desync_attack(sources, target, av, used):
    """
    Try to send 2-3 waves to arrive simultaneously.
    Returns list of moves if feasible, else [].
    """
    if len(sources) < 2: return []
    
    # Calculate arrival time from each source with minimum ships
    arrivals = []
    for src in sources[:3]:
        spare = src.ships - used.get(src.id,0) - 3
        if spare < 4: continue
        _,dd,eta = icp(src.x,src.y,target,av,spare)
        arrivals.append((eta, src, spare, dd))
    
    if len(arrivals) < 2: return []
    
    # Find target arrival time = median
    arrivals.sort(key=lambda x: x[0])
    target_eta = arrivals[len(arrivals)//2][0]
    
    # For each source, compute ships needed to arrive at target_eta
    # If eta < target_eta: send fewer ships (slower fleet)
    # If eta > target_eta: can't slow down, skip
    moves = []
    total_sent = 0
    garrison_arr = target.ships + target.production * target_eta
    
    for eta, src, spare, dd in arrivals:
        if eta > target_eta + 3: continue  # too far, skip
        
        # Ships to send from this source
        frac = spare // len(arrivals)
        n = max(4, frac)
        if src.ships - used.get(src.id,0) - n < 3: continue
        
        a2,dd2,_ = icp(src.x,src.y,target,av,n)
        sa,ok = safe(src.x,src.y,a2,dd2)
        if not ok: continue
        
        moves.append((src.id, sa, n))
        total_sent += n
    
    # Only return if combined force is enough to capture
    if total_sent > garrison_arr * 1.05 and len(moves) >= 2:
        return moves
    return []

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 2: ECONOMIC SUFFOCATION
# Send tiny fleets (2-4 ships) to arrive at enemy high-production planets
# exactly when their production counter resets → forces them to keep garrison
# Result: enemy planet never accumulates ships to attack us
# ─────────────────────────────────────────────────────────────────────────────
def suffocation_targets(enemy_planets, mine, av, used, step):
    """
    Find enemy planets worth suffocating.
    Returns list of (source, target, n_ships, safe_angle)
    """
    if step < 60: return []  # don't suffocate in early game
    
    moves = []
    high_prod = sorted([p for p in enemy_planets if p.production >= 3],
                       key=lambda p: -p.production)[:2]
    
    for tgt in high_prod:
        # Find closest source with spare ships
        best_src = None; best_dd = 1e9
        for src in mine:
            spare = src.ships - used.get(src.id,0) - 5
            if spare < 3: continue
            dd = d2(src.x,src.y,tgt.x,tgt.y)
            if dd < best_dd:
                best_dd = dd; best_src = src
        
        if best_src is None: continue
        
        # Send just 3 ships — enough to force garrison, not enough to capture
        # Arrive every ~15 turns (production cycle)
        if step % 15 != 0 and step % 15 != 1: continue
        
        n = 3
        a,dd,_ = icp(best_src.x,best_src.y,tgt,av,n)
        sa,ok = safe(best_src.x,best_src.y,a,dd)
        if ok and best_src.ships - used.get(best_src.id,0) - n >= 5:
            moves.append((best_src.id, sa, n))
    
    return moves

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 3: SNIPE PROTOCOL
# Detect enemy fleets heading to neutral planets
# Send our fleet to arrive 1-2 turns AFTER enemy lands → they weakened it for us
# ─────────────────────────────────────────────────────────────────────────────
def find_snipe_opportunities(neutral_planets, enemy_fleets, mine, av, used):
    """
    For each neutral planet with an incoming enemy fleet:
    - Estimate enemy arrival time
    - Estimate post-battle state (enemy ships - garrison)
    - If we can arrive 1-3 turns after with enough to capture → SNIPE
    """
    snipes = []
    
    for neutral in neutral_planets:
        # Find enemy fleets heading here
        incoming = []
        for f in enemy_fleets:
            _,dd,eta = icp(f.x,f.y,neutral,av,f.ships)
            if dd < neutral.radius + 4 and eta < 50:
                incoming.append((eta, f.ships))
        
        if not incoming: continue
        
        incoming.sort(key=lambda x: x[0])
        enemy_eta, enemy_ships = incoming[0]
        
        # After battle: neutral has (neutral.ships + prod*eta) - enemy_ships survivors
        # If enemy_ships > neutral, enemy captures it with leftover ships
        after_neutral = neutral.ships + neutral.production * enemy_eta
        if enemy_ships <= after_neutral:
            continue  # enemy loses, neutral survives at full strength — skip
        
        # Enemy captures neutral with (enemy_ships - after_neutral) ships
        enemy_leftover = max(1, enemy_ships - after_neutral)
        
        # We arrive 2 turns later → need to beat enemy_leftover + 2 turns production
        snipe_eta = enemy_eta + 2
        ships_needed = int(enemy_leftover * 1.1) + neutral.production * 2 + 2
        
        # Find best source
        for src in sorted(mine, key=lambda p: d2(p.x,p.y,neutral.x,neutral.y)):
            spare = src.ships - used.get(src.id,0) - 4
            if spare < ships_needed: continue
            
            a,dd,our_eta = icp(src.x,src.y,neutral,av,ships_needed)
            # We need our_eta ≈ snipe_eta (arrive after enemy)
            if our_eta < enemy_eta + 0.5: continue  # we'd arrive too early
            if our_eta > enemy_eta + 8:   continue  # too late
            
            sa,ok = safe(src.x,src.y,a,dd)
            if ok:
                score = neutral.production * 5 + (enemy_ships - enemy_leftover)
                snipes.append((score, src.id, sa, ships_needed, neutral.id))
                break
    
    snipes.sort(key=lambda x: -x[0])
    return snipes

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 4: PRE-EMPTIVE STRIKE
# If enemy planet has > 80% of ships it had 5 turns ago → it's massing
# Attack NOW before they launch → destroy ships in harbor
# ─────────────────────────────────────────────────────────────────────────────
_ship_history = collections.defaultdict(list)  # planet_id -> [ships_count]

def update_history(planets):
    for p in planets:
        if p.owner >= 0:
            _ship_history[p.id].append(p.ships)
            if len(_ship_history[p.id]) > 8:
                _ship_history[p.id].pop(0)

def find_massing_planets(enemy_planets, mine, av, used):
    """
    Detect enemy planets rapidly accumulating ships.
    Returns (planet, urgency) pairs sorted by urgency.
    """
    massing = []
    for p in enemy_planets:
        hist = _ship_history.get(p.id, [])
        if len(hist) < 4: continue
        
        growth = p.ships - hist[-4]  # ships gained in last 4 turns
        expected_growth = p.production * 4
        
        # If growing faster than production alone → they're NOT attacking → massing
        if growth > expected_growth * 0.7 and p.ships > 30:
            urgency = p.ships * p.production
            massing.append((p, urgency))
    
    massing.sort(key=lambda x: -x[1])
    return massing

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 5: MULTI-ENEMY AWARENESS (improved dominance)
# Count nearby enemy power, not just global strongest
# ─────────────────────────────────────────────────────────────────────────────
def compute_status(planets, fleets, mine, player):
    """
    Returns ('winning'/'losing', aggression_level 0-1)
    aggression_level: 0 = turtle, 1 = all-in
    """
    if not mine: return 'losing', 1.0
    
    my_cx = sum(p.x for p in mine)/len(mine)
    my_cy = sum(p.y for p in mine)/len(mine)
    
    my_ships = sum(p.ships for p in mine)
    my_prod  = sum(p.production for p in mine)
    my_fleet = sum(f.ships for f in fleets if f.owner==player)
    my_power = my_ships + my_prod*25 + my_fleet
    
    # Sum nearby enemy power (within 45 units of our centroid)
    nearby_enemy_power = 0
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        proximity = max(0, 1 - d2(p.x,p.y,my_cx,my_cy)/60)
        nearby_enemy_power += (p.ships + p.production*25) * (0.5+proximity)
    
    # Add enemy fleets heading toward us
    for f in fleets:
        if f.owner==player or f.owner<0: continue
        # Check if heading roughly toward our centroid
        dx,dy = math.cos(f.angle),math.sin(f.angle)
        dot = (my_cx-f.x)*dx + (my_cy-f.y)*dy
        if dot > 0:
            nearby_enemy_power += f.ships * 0.8
    
    ratio = my_power / max(1, nearby_enemy_power)
    
    if ratio >= 1.15:
        return 'winning', 0.5   # winning: moderate aggression
    elif ratio >= 0.85:
        return 'even', 0.75     # even: push harder
    else:
        return 'losing', 1.0    # losing: all-in, break their economy

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 6: ORBITAL PHASE WINDOWS
# ─────────────────────────────────────────────────────────────────────────────
def phase_mult(planet, mine, av):
    if not inn(planet) or not mine: return 1.0
    r  = d2(planet.x,planet.y,SX,SY)
    a0 = math.atan2(planet.y-SY,planet.x-SX)
    a15= a0+av*15
    fx = SX+r*math.cos(a15); fy=SY+r*math.sin(a15)
    cx = sum(p.x for p in mine)/len(mine)
    cy = sum(p.y for p in mine)/len(mine)
    df = d2(fx,fy,cx,cy); dc=d2(planet.x,planet.y,cx,cy)
    if df < dc*0.85: return 1.65
    if df > dc*1.15: return 0.72
    return 1.0

# ─────────────────────────────────────────────────────────────────────────────
# WEAPON 7: RETROGRADE WARFARE
# ─────────────────────────────────────────────────────────────────────────────
def retro_targets(planets, fleets, player):
    retro = []
    for p in planets:
        if p.owner<0 or p.owner==player: continue
        departed = sum(f.ships for f in fleets
                       if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<12 and f.ships>0)
        if departed < 8: continue
        ratio = departed/max(1,p.ships+departed)
        if ratio >= 0.28:
            retro.append((p, ratio*p.production*4+departed*0.3))
    retro.sort(key=lambda x:-x[1])
    return retro

# ─────────────────────────────────────────────────────────────────────────────
# GARRISON
# ─────────────────────────────────────────────────────────────────────────────
def garrison(planet, status, aggression, incoming=0):
    if incoming>0: return int(incoming*1.12)+5
    base = max(3, planet.production*2)
    if status=='winning':   return base
    if aggression > 0.85:   return max(3, base//2)  # losing: bare minimum
    return base

# ─────────────────────────────────────────────────────────────────────────────
# MAIN AGENT
# ─────────────────────────────────────────────────────────────────────────────
def orbital_strategist(obs):
    global _ship_history
    
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    = [p for p in planets if p.owner==pl]
    neutral = [p for p in planets if p.owner<0]
    enemy   = [p for p in planets if p.owner>=0 and p.owner!=pl]
    others  = enemy+neutral

    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()

    def avail(p): return p.ships-used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n

    # ── Update massing history ─────────────────────────────────────────────
    update_history(planets)

    # ── Compute status ─────────────────────────────────────────────────────
    status, aggr = compute_status(planets, fleets, mine, pl)

    # ── Incoming threats ───────────────────────────────────────────────────
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_ = icp(f.x,f.y,p,av,f.ships)
            if dd < p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships

    # ── DEFENSE ───────────────────────────────────────────────────────────
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=garrison(p,status,aggr,thr); deficit=need-avail(p)
        if deficit<=0: continue
        donors=sorted([s for s in mine if s.id!=p.id and
                       avail(s)-garrison(s,status,aggr)>4],
                      key=lambda s:d2(s.x,s.y,p.x,p.y))
        for src in donors[:3]:
            send=min(avail(src)-garrison(src,status,aggr),deficit)
            if send<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,send); sa,ok=safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,send]); rsv(src.id,send); deficit-=send
            if deficit<=0: break

    # ── En-route targets ───────────────────────────────────────────────────
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<70: enroute.add(t.id)

    enemy_fleets=[f for f in fleets if f.owner!=pl and f.owner>=0]

    # ── WEAPON 3: SNIPE PROTOCOL ──────────────────────────────────────────
    snipes=find_snipe_opportunities(
        [t for t in neutral if t.id not in done and t.id not in enroute],
        enemy_fleets, mine, av, used)
    for score,sid,sa,n,tid in snipes[:2]:
        src=next((p for p in mine if p.id==sid),None)
        if src is None or avail(src)<n+garrison(src,status,aggr): continue
        moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── WEAPON 4: PRE-EMPTIVE STRIKE ──────────────────────────────────────
    massing=find_massing_planets(enemy,mine,av,used)
    for tgt,urg in massing[:1]:
        if tgt.id in done or tgt.id in enroute: continue
        best_src=max(
            [p for p in mine if avail(p)-garrison(p,status,aggr)>8],
            key=lambda p:avail(p)-garrison(p,status,aggr),default=None)
        if best_src is None: continue
        spare=avail(best_src)-garrison(best_src,status,aggr)
        _,_,eta=icp(best_src.x,best_src.y,tgt,av,spare)
        n=max(int((tgt.ships+tgt.production*eta)*1.08)+1,tgt.ships+3)
        if spare<n: continue
        a,dd,_=icp(best_src.x,best_src.y,tgt,av,n)
        sa,ok=safe(best_src.x,best_src.y,a,dd)
        if ok:
            moves.append([best_src.id,sa,n]); rsv(best_src.id,n); done.add(tgt.id)

    # ── WEAPON 7: RETROGRADE ──────────────────────────────────────────────
    for rp_t,rscore in retro_targets(planets,fleets,pl)[:2]:
        if rp_t.id in done or rp_t.id in enroute: continue
        best_src=None; best_n=0; best_sa=0.0; best_dd=1e9
        for src in mine:
            spare=avail(src)-garrison(src,status,aggr)
            if spare<4: continue
            _,_,eta=icp(src.x,src.y,rp_t,av,spare)
            n=max(int((rp_t.ships+rp_t.production*eta)*1.06)+1,rp_t.ships+2)
            if spare<n: continue
            a2,dd2,_=icp(src.x,src.y,rp_t,av,n); sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            if best_src is None or dd2<best_dd:
                best_src,best_n,best_sa,best_dd=src,n,sa,dd2
        if best_src and avail(best_src)-garrison(best_src,status,aggr)>=best_n:
            moves.append([best_src.id,best_sa,best_n])
            rsv(best_src.id,best_n); done.add(rp_t.id)

    # ── WEAPON 1: DESYNC ATTACK on high-value targets ─────────────────────
    high_value=[t for t in enemy if t.id not in done and t.id not in enroute
                and t.production>=3 and t.ships>20]
    for tgt in sorted(high_value,key=lambda t:-t.production*t.ships)[:1]:
        srcs=[p for p in mine if avail(p)-garrison(p,status,aggr)>6]
        if len(srcs)>=2:
            dm=desync_attack(srcs,tgt,av,used)
            if dm:
                for sid,sa,n in dm:
                    moves.append([sid,sa,n]); rsv(sid,n)
                done.add(tgt.id)

    # ── WEAPON 2: SUFFOCATION ─────────────────────────────────────────────
    suf=suffocation_targets(enemy,mine,av,used,stp)
    for sid,sa,n in suf:
        src=next((p for p in mine if p.id==sid),None)
        if src and avail(src)>=n+garrison(src,status,aggr):
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── MAIN SCORING: all remaining targets ───────────────────────────────
    candidates=[]
    for src in mine:
        spare=avail(src)-garrison(src,status,aggr)
        if spare<4: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            _,_,eta=icp(src.x,src.y,tgt,av,max(spare,5))
            buf=1.08 if tgt.owner>=0 else 1.05
            n=max(int((tgt.ships+tgt.production*eta)*buf)+1,int(tgt.ships*buf)+2)
            if n>spare: continue
            a2,dd2,eta2=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            tw=max(0,rem-eta2); prod=tgt.production
            score=(prod**2)*10*tw+prod*tw
            score*=phase_mult(tgt,mine,av)
            if tgt.owner>=0:
                score*=1.5
                e_prod=sum(p.production for p in planets if p.owner==tgt.owner)
                if e_prod>sum(p.production for p in mine)*1.1: score*=1.3
            if tgt.ships<=tgt.production*2+3: score*=1.9
            score-=dd2*0.4+n*0.25
            if status=='losing': score=score*1.4 if prod>=3 else score*0.6
            candidates.append((score,src,tgt,n,sa,dd2))

    candidates.sort(key=lambda x:-x[0])
    max_atk=5 if (stp<90 or status=='losing') else 4
    atks=0
    for score,src,tgt,n,sa,dd in candidates:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        spare=avail(src)-garrison(src,status,aggr)
        if spare<n: continue
        moves.append([src.id,sa,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ─────────────────────────────────────────────
    for src in sorted(mine,key=lambda p:-avail(p)):
        spare=avail(src)-garrison(src,status,aggr)
        if spare<5: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            _,_,eta=icp(src.x,src.y,tgt,av,spare)
            buf=1.08 if tgt.owner>=0 else 1.05
            n=max(int((tgt.ships+tgt.production*eta)*buf)+1,int(tgt.ships*buf)+2)
            if n>spare: continue
            a2,dd2,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a2,dd2)
            if not ok: continue
            sc=(tgt.production**2)/(dd2+1)*phase_mult(tgt,mine,av)+spare*0.05
            if sc>bsc: bsc=sc; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## ✅ Cell 9 — Verify Submission


In [ ]:
import importlib.util
_ship_history = _c.defaultdict(list)
spec=importlib.util.spec_from_file_location('main','main.py')
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
sub=mod.agent
print(f'✅ main.py OK — agent: {sub.__name__}')
ev=make('orbit_wars',debug=False)
ev.run([sub,v1_agent,'random',v1_agent])
fr=[s.reward for s in ev.steps[-1]]
print(f'Rewards: {fr}')
print('🏆 WINS!' if fr[0]==1 else '✅ Runs correctly')
